In [ ]:
import os
import json
import uuid
import cv2 as cv
import numpy as np
from pathlib import Path
import matplotlib.pyplot as plt

# --- Constants & Paths ---
RAW_DATA_DIR = "data/raw/easy"
OUTPUT_JSON = "label_studio_import.json"
# You need a web server or local path reachable by Label Studio
# If running locally, you often use "Local Storage" in Label Studio
IMAGE_URL_PREFIX = "http://localhost:8080/data/" 

def create_ls_prediction(classified_symbols, img_w, img_h):
    """Converts classified symbols to Label Studio result format."""
    results = []
    for s in classified_symbols:
        x, y, w, h = s.bbox
        
        # Label Studio uses percentages (0-100) for coordinates
        ls_x = (x / img_w) * 100
        ls_y = (y / img_h) * 100
        ls_w = (w / img_w) * 100
        ls_h = (h / img_h) * 100

        results.append({
            "id": str(uuid.uuid4())[:8],
            "type": "rectanglelabels",        
            "from_name": "label",              # Must match your Label Studio config
            "to_name": "image",
            "original_width": img_w,
            "original_height": img_h,
            "image_rotation": 0,
            "value": {
                "rotation": 0,                 # Default 0, you will fix this in LS
                "x": ls_x,
                "y": ls_y,
                "width": ls_w,
                "height": ls_h,
                "rectanglelabels": [s.label]
            },
            "score": float(s.confidence)
        })
    return results

def find_stitch_roi(binary: np.ndarray,
                    min_fill_ratio: float = 0.05,
                    margin_frac: float = 0.02) -> tuple[int,int,int,int]:
    """
    Locate the bounding box of the densest connected region (the stitch grid).
    Returns (x, y, w, h) in pixel coordinates.

    Strategy
    --------
    1. Dilate heavily so nearby symbols merge into blobs.
    2. Find the largest blob by area — that is the stitch grid.
    3. Return its bounding rectangle with a small margin.
    """
    kernel = cv.getStructuringElement(cv.MORPH_ELLIPSE, (2, 2))
    cleaned = cv.morphologyEx(binary, cv.MORPH_OPEN, kernel, iterations=1)

    # 4. Close small gaps inside symbols
    kernel2 = cv.getStructuringElement(cv.MORPH_ELLIPSE, (3, 3))
    closed = cv.morphologyEx(cleaned, cv.MORPH_CLOSE, kernel2, iterations=1)


    # Dilate so all stitch symbols within a piece connect
    k = cv.getStructuringElement(cv.MORPH_RECT, (30, 20))
    dilated = cv.dilate(closed, k, iterations=3)
    
    # Find contours of merged blobs
    contours, _ = cv.findContours(dilated, cv.RETR_EXTERNAL,
                                    cv.CHAIN_APPROX_SIMPLE)
    if not contours:
        h, w = closed.shape
        return 0, 0, w, h

    # Pick the largest contour
    largest = max(contours, key=cv.contourArea)
    x, y, w, h = cv.boundingRect(largest)

    # Add margin
    H, W = closed.shape
    mx = int(W * margin_frac)
    my = int(H * margin_frac)
    x = max(0, x - mx);  y = max(0, y - my)
    w = min(W - x, w + 2*mx)
    h = min(H - y, h + 2*my)
    return x, y, w, h

def count_rows_from_contours(contours, gap_factor=0.5):
    """
    Count stitch rows by clustering contour bounding boxes by their y-position.

    Contours belonging to the same row have similar y-centres.
    A gap larger than gap_factor * median_height between consecutive
    y-centres means we've crossed into a new row.

    Args:
        contours:    list of OpenCV contours
        gap_factor:  gap threshold as a fraction of the median symbol height.
                     Increase if rows are being merged; decrease if one row
                     is being split into multiple.

    Returns:
        n_rows:      number of detected rows
        row_groups:  list of lists — each inner list holds the contour indices
                     belonging to that row, top to bottom
    """
    if not contours:
        return 0, []

    # Collect (y_centre, height, original_index) for every contour
    boxes = []
    for i, c in enumerate(contours):
        x, y, w, h = cv.boundingRect(c)
        y_centre = y + h / 2
        boxes.append((y_centre, h, i))

    # Sort by y_centre top → bottom
    boxes.sort(key=lambda b: b[0])
    y_centres = np.array([b[0] for b in boxes])
    heights   = np.array([b[1] for b in boxes])

    median_h  = np.median(heights)
    min_gap   = gap_factor * median_h

    # Gap-based clustering: a jump larger than min_gap starts a new row
    row_groups = []
    current_row = [boxes[0][2]]

    for i in range(1, len(boxes)):
        gap = y_centres[i] - y_centres[i - 1]
        if gap > min_gap:
            row_groups.append(current_row)
            current_row = []
        current_row.append(boxes[i][2])
    row_groups.append(current_row)

    n_rows = len(row_groups)

    # ── Visualisation ────────────────────────────────────────────────
    colors = plt.cm.tab10.colors
    overlay = cv.cvtColor(roi_rgb.copy(), cv.COLOR_BGR2RGB)

    for row_idx, group in enumerate(row_groups):
        color_bgr = tuple(int(c * 255) for c in colors[row_idx % len(colors)][:3])
        for ci in group:
            x, y, w, h = cv.boundingRect(contours[ci])
            cv.rectangle(overlay, (x, y), (x + w, y + h), color_bgr[::-1], 2)
            cv.putText(overlay, str(row_idx + 1), (x, y - 4),
                       cv.FONT_HERSHEY_SIMPLEX, 0.4, color_bgr[::-1], 1)

    plt.figure(figsize=(10, 10))
    plt.imshow(overlay)
    plt.title(f"{n_rows} row(s) detected  |  {len(contours)} contours total")
    plt.axis("off")
    plt.tight_layout()
    plt.show()

    for i, group in enumerate(row_groups):
        print(f"  Row {i + 1}: {len(group)} symbol(s)")

    return n_rows, row_groups


# --- Main Loop ---
ls_data = []
image_files = [f for f in os.listdir(RAW_DATA_DIR) if f.endswith(('.png', '.jpg', '.jpeg'))]

print(f"Processing {len(image_files)} images...")

for filename in image_files:
    path = os.path.join(RAW_DATA_DIR, filename)
    img = cv.imread(path)
    if img is None: continue
    
    h_orig, w_orig = img.shape[:2]
    gray = cv.cvtColor(img, cv.COLOR_BGR2GRAY)
    
    # 1. Reuse your thresholding logic
    denoised = cv.fastNlMeansDenoising(gray, h=10)
    thresh = cv.adaptiveThreshold(denoised, 255, cv.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                  cv.THRESH_BINARY_INV, 19, 4)
    
    # 2. Find ROI and Symbols
    roi = find_stitch_roi(thresh)
    roi_x, roi_y, roi_w, roi_h = roi
    roi_gray = gray[roi_y:roi_y+roi_h, roi_x:roi_x+roi_w]
    roi_bin = thresh[roi_y:roi_y+roi_h, roi_x:roi_x+roi_w]
    
    # Filter/Find contours (Reuse your nested filter logic here)
    cnts, _ = cv.findContours(roi_bin, cv.RETR_TREE, cv.CHAIN_APPROX_SIMPLE)
    # ... (Insert your filtered_indices / row_groups logic from your notebook here) ...
    # For brevity, I'm assuming n_rows and row_groups are calculated as per your code
    n_rows, row_groups = count_rows_from_contours(cnts) 
    
    # 3. Classify
    current_classified = []
    for sym in iter_symbols(roi_gray, cnts, row_groups):
        label = clf.predict(sym.image)
        proba = clf.predict_proba(sym.image)
        
        # Adjust bbox back to full image coordinates
        abs_x = sym.bbox[0] + roi_x
        abs_y = sym.bbox[1] + roi_y
        
        current_classified.append(ClassifiedSymbol(
            image=None, row=sym.row, col=sym.col,
            bbox=(abs_x, abs_y, sym.bbox[2], sym.bbox[3]),
            label=label, confidence=proba[label]
        ))
    
    # 4. Create Label Studio Task
    task = {
        "data": {
            "image": f"{IMAGE_URL_PREFIX}{filename}" 
        },
        "predictions": [{
            "model_version": "mobilenet_v2_v1",
            "result": create_ls_prediction(current_classified, w_orig, h_orig)
        }]
    }
    ls_data.append(task)

# Save to file
with open(OUTPUT_JSON, 'w') as f:
    json.dump(ls_data, f, indent=2)

print(f"Success! Import '{OUTPUT_JSON}' into Label Studio.")

Processing 6 images...


NameError: name 'find_stitch_roi' is not defined